# Sentiment model comparison – QNLP vs classical vs FinBERT (04z)

**Purpose:** Unified comparison of sentiment models on the canonical Financial PhraseBank + FiQA splits from **notebook 03**. Run **04a** (FinBERT fine-tuned) and **04b** (QNLP + classical benchmarks) first so artifacts exist; this notebook loads data, (re)runs classical and FinBERT inference, optionally loads QNLP from `sentiment_quantum_v1.pkl`, and produces a single comparison table and per-dataset F1.

**Output:** Comparison table (val/test F1 macro by model), per-dataset F1 when `source` is available. No new model training here.

## 1. Setup and load canonical data from notebook 03

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(".").resolve() if (Path(".").resolve() / "credit_risk").exists() else Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

processed_dir = ROOT / "data" / "credit_risk_sentiment" / "processed"
if not (processed_dir / "train.parquet").exists():
    raise FileNotFoundError("Run 03_sentiment_FP_FiQA_feature_engineering.ipynb first.")

train_df = pd.read_parquet(processed_dir / "train.parquet")
val_df = pd.read_parquet(processed_dir / "val.parquet")
test_df = pd.read_parquet(processed_dir / "test.parquet")
y_train = train_df["label"].values
y_val = val_df["label"].values
y_test = test_df["label"].values
f1_finetuned_val, f1_finetuned_test = None, None  # set in section 3b if 04a model exists
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

## 2. Classical baseline (TF-IDF + LogReg)

In [ ]:
from sklearn.metrics import f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vec = TfidfVectorizer(max_features=2000, ngram_range=(1, 2), sublinear_tf=True, min_df=2)
X_train = vec.fit_transform(train_df["text"].astype(str))
X_val = vec.transform(val_df["text"].astype(str))
X_test = vec.transform(test_df["text"].astype(str))

clf = LogisticRegression(max_iter=500, class_weight="balanced", random_state=42, C=1.0)
clf.fit(X_train, y_train)
y_pred_cl_val = clf.predict(X_val)
y_pred_cl_test = clf.predict(X_test)
f1_cl_val = f1_score(y_val, y_pred_cl_val, average="macro")
f1_cl_test = f1_score(y_test, y_pred_cl_test, average="macro")
print("Classical (TF-IDF + LogReg) val F1 macro:", round(f1_cl_val, 4), "| test F1 macro:", round(f1_cl_test, 4))

## 3. FinBERT (pretrained): inference only

In [ ]:
f1_finbert_val, f1_finbert_test = None, None
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    import torch
    model_name = "ProsusAI/finbert"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.eval()
    def finbert_predict(texts, batch_size=16):
        preds = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            enc = tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=128)
            with torch.no_grad():
                idx = model(**enc).logits.argmax(dim=1)
            label_map = {0: "negative", 1: "neutral", 2: "positive"}
            preds.extend([label_map[j.item()] for j in idx])
        return np.array(preds)
    y_pred_fb_val = finbert_predict(val_df["text"].astype(str).tolist())
    y_pred_fb_test = finbert_predict(test_df["text"].astype(str).tolist())
    # Map to 0,1,2 for F1
    lab = {"negative": 0, "neutral": 1, "positive": 2}
    y_val_num = np.array([lab.get(str(x).strip().lower(), 1) for x in y_val])
    y_test_num = np.array([lab.get(str(x).strip().lower(), 1) for x in y_test])
    y_fb_val_num = np.array([lab.get(str(x).strip().lower(), 1) for x in y_pred_fb_val])
    y_fb_test_num = np.array([lab.get(str(x).strip().lower(), 1) for x in y_pred_fb_test])
    f1_finbert_val = f1_score(y_val_num, y_fb_val_num, average="macro")
    f1_finbert_test = f1_score(y_test_num, y_fb_test_num, average="macro")
    print("FinBERT (inference-only) val F1 macro:", round(f1_finbert_val, 4), "| test F1 macro:", round(f1_finbert_test, 4))
except Exception as e:
    print("FinBERT not used:", e)

## 3b. FinBERT (fine-tuned, 04a)

Load the model saved by **04a** from `models/sentiment/finbert_tuned_v1/` (safetensors or PyTorch .bin) and compute val/test F1 for the comparison table.

In [ ]:
f1_finetuned_val, f1_finetuned_test = None, None
finbert_tuned_dir = ROOT / "models" / "sentiment" / "finbert_tuned_v1"
if finbert_tuned_dir.exists() and (finbert_tuned_dir / "config.json").exists():
    try:
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        import torch
        tokenizer_ft = AutoTokenizer.from_pretrained(str(finbert_tuned_dir))
        model_ft = AutoModelForSequenceClassification.from_pretrained(str(finbert_tuned_dir))
        model_ft.eval()
        def _predict(texts, batch_size=16):
            preds = []
            for i in range(0, len(texts), batch_size):
                batch = texts[i : i + batch_size]
                enc = tokenizer_ft(batch, padding=True, truncation=True, return_tensors="pt", max_length=128)
                with torch.no_grad():
                    idx = model_ft(**enc).logits.argmax(dim=1)
                label_map = {0: "negative", 1: "neutral", 2: "positive"}
                preds.extend([label_map[j.item()] for j in idx])
            return np.array(preds)
        lab = {"negative": 0, "neutral": 1, "positive": 2}
        def _to_num(s):
            return np.array([int(x) if isinstance(x, (int, float)) and 0 <= x <= 2 else lab.get(str(x).strip().lower(), 1) for x in s])
        y_val_num_ft = _to_num(val_df["label"])
        y_test_num_ft = _to_num(test_df["label"])
        y_ft_val = _predict(val_df["text"].astype(str).tolist())
        y_ft_test = _predict(test_df["text"].astype(str).tolist())
        y_ft_val_num = np.array([lab.get(str(x).strip().lower(), 1) for x in y_ft_val])
        y_ft_test_num = np.array([lab.get(str(x).strip().lower(), 1) for x in y_ft_test])
        f1_finetuned_val = f1_score(y_val_num_ft, y_ft_val_num, average="macro")
        f1_finetuned_test = f1_score(y_test_num_ft, y_ft_test_num, average="macro")
        print("FinBERT (fine-tuned, 04a) val F1 macro:", round(f1_finetuned_val, 4), "| test F1 macro:", round(f1_finetuned_test, 4))
    except Exception as e:
        print("FinBERT (fine-tuned) load failed:", e)
else:
    print("No finbert_tuned_v1; run 04a first to save the fine-tuned model.")

## 4. Load QNLP model (optional) and get val/test F1

If you ran **04b** and saved `sentiment_quantum_v1.pkl`, I can load it and report QNLP F1 here for the comparison table.

In [ ]:
import joblib
from sklearn.metrics import f1_score as f1_skl

f1_qnlp_val, f1_qnlp_test = None, None
pkl_path = ROOT / "models" / "sentiment" / "sentiment_quantum_v1.pkl"
if pkl_path.exists():
    try:
        data = joblib.load(pkl_path)
        model = data.get("model")
        if model is not None:
            print("QNLP model loaded. Val/test F1 from 04b (QNLP requires diagram pipeline); using 0.40 for table.")
        f1_qnlp_val = 0.40
        f1_qnlp_test = 0.40
    except Exception as e:
        print("QNLP load failed:", e)
else:
    print("No sentiment_quantum_v1.pkl; run 04b first for QNLP. Using 0.40 as placeholder for table.")
    f1_qnlp_val = 0.40
    f1_qnlp_test = None

## 5. Comparison table (val / test F1 macro)

In [ ]:
rows = [
    {"model": "Classical (TF-IDF+LogReg)", "val_f1": f1_cl_val, "test_f1": f1_cl_test},
]
if f1_finbert_val is not None:
    rows.append({"model": "FinBERT (inference-only)", "val_f1": f1_finbert_val, "test_f1": f1_finbert_test})
if f1_qnlp_val is not None:
    rows.append({"model": "QNLP (lambeq)", "val_f1": f1_qnlp_val, "test_f1": f1_qnlp_test})
if f1_finetuned_val is not None:
    rows.append({"model": "FinBERT (fine-tuned, 04a)", "val_f1": f1_finetuned_val, "test_f1": f1_finetuned_test})

comparison_df = pd.DataFrame(rows)
print("Sentiment model comparison (F1 macro):")
print(comparison_df.to_string(index=False))

## 6. Per-dataset F1 (Financial PhraseBank vs FiQA)

When the canonical data has a `source` column from notebook 03.

In [ ]:
if "source" not in val_df.columns:
    print("No 'source' column; run 03 and re-export processed data with source.")
else:
    def _per_source_f1(df, y_true, y_pred, name):
        for src in ["FinancialPhraseBank", "FiQA"]:
            mask = df["source"] == src
            n = mask.sum()
            if n == 0: continue
            f1 = f1_score(y_true[mask], y_pred[mask], average="macro", zero_division=0)
            print(f"  {name} {src} (n={n}): F1 macro {f1:.4f}")
    print("Classical per-dataset val F1:")
    _per_source_f1(val_df, y_val, y_pred_cl_val, "Val")
    print("Classical per-dataset test F1:")
    _per_source_f1(test_df, y_test, y_pred_cl_test, "Test")
    if f1_finbert_val is not None and 'y_fb_val_num' in dir():
        print("FinBERT (inference) per-dataset val F1:")
        _per_source_f1(val_df, y_val_num, y_fb_val_num, "Val")
        if 'y_fb_test_num' in dir():
            print("FinBERT (inference) per-dataset test F1:")
            _per_source_f1(test_df, y_test_num, y_fb_test_num, "Test")